# Proyek Klasifikasi Email Spam
## Notebook 1: Exploratory Data Analysis (EDA) & Data Preprocessing
Tujuan dari notebook ini adalah memuat data mentah Enron Spam, mengeksplorasi strukturnya, dan membersihkan teks sebelum masuk ke tahap ekstraksi fitur.

In [1]:
# Menghubungkan Colab dengan Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import numpy as np

### 1. Load Data Mentah
Mengambil file `enron_spam_data.csv` dari direktori Google Drive.

In [3]:
file_path = '/content/drive/MyDrive/Semester 4/Machine Learning/Final Project/data/enron_spam_data.csv'

# Membaca file CSV menjadi DataFrame Pandas
df = pd.read_csv(file_path)
df.head()

,Unnamed: 0,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14


### 2. Inspeksi Awal Data (Sanity Check)
Mengecek dimensi data, tipe kolom, dan mendeteksi adanya *missing value* (data kosong).

In [4]:
# Cek total baris dan kolom
print("Dimensi dataset:", df.shape)
print("-" * 30)

# Cek informasi detail tiap kolom
df.info()

Dimensi dataset: (33716, 5)
------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33716 entries, 0 to 33715
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  33716 non-null  int64 
 1   Subject     33716 non-null  object
 2   Message     33664 non-null  object
 3   Spam/Ham    33716 non-null  object
 4   Date        33716 non-null  object
dtypes: int64(1), object(4)
memory usage: 1.3+ MB


### 3. Data Cleaning (Pembersihan Struktur & Missing Values)
Membuang kolom yang tidak relevan (`Unnamed: 0` dan `Date`) serta menghapus baris yang isi pesannya (kolom `Message`) kosong / NaN.

In [5]:
# 1. Membuang kolom yang tidak dibutuhkan (axis=1 berarti membuang kolom)
df_cleaned = df.drop(['Unnamed: 0', 'Date'], axis=1)

# 2. Cek missing value pada kolom 'Message'
print("Jumlah missing value pada kolom 'Message':")
print(df_cleaned['Message'].isna().sum())

# Menampilkan baris yang memiliki missing value pada kolom 'Message'
print("\nBaris dengan missing value pada kolom 'Message':")
print(df_cleaned[df_cleaned['Message'].isna()])

# (Opsional) Melihat persentase missing value
persentase = (df_cleaned['Message'].isna().sum() / len(df_cleaned)) * 100
print(f"\nPersentase missing value: {persentase:.2f}%")

# 2. Menghapus baris yang kolom 'Message'-nya kosong (NaN)
# df_cleaned = df_cleaned.dropna(subset=['Message'])

# print("Dimensi setelah pembersihan awal:", df_cleaned.shape)

Jumlah missing value pada kolom 'Message':
52

Baris dengan missing value pada kolom 'Message':
                                                 Subject Message Spam/Ham
0                           christmas tree farm pictures     NaN      ham
183    day 26 - txu lonestar called on 20000 at carthage     NaN      ham
199    is this fri feb 11 a problem for taking vacati...     NaN      ham
304    http : / / www . pge - texas . com / www / gtt...     NaN      ham
1655                                             revised     NaN      ham
2537                                      house pictures     NaN      ham
3271   imperial sugar ' s volumes will be 142 , 000 -...     NaN      ham
3273                                 gymnastics pictures     NaN      ham
3280   i received a fax from cp & l for june 2001 . t...     NaN      ham
3345   http : / / 208 . 246 . 87 . 65 / info _ postin...     NaN      ham
3563   here ' s the list , dirty , but it ' s a list ...     NaN      ham
5269            

### 4. Menangani Data Duplikat
Mengecek dan menghapus email yang isinya sama persis untuk mencegah kebocoran data (data leakage) saat melatih model.

In [6]:
# Mengecek jumlah data duplikat
jumlah_duplikat = df_cleaned.duplicated().sum()
print(f"Ditemukan {jumlah_duplikat} baris data duplikat.")

# Menghapus duplikat dan mereset ulang urutan index
# df_cleaned = df_cleaned.drop_duplicates().reset_index(drop=True) -> NONAKTIFKAN BARIS INI AGAR SPAM TIDAK PUNAH

print("Dimensi data saat ini:", df_cleaned.shape)

Ditemukan 17800 baris data duplikat.
Dimensi data saat ini: (33716, 3)


### 5. Menggabungkan Fitur Teks
Menggabungkan kolom `Subject` dan `Message` agar model bisa belajar dari keseluruhan konteks email, lalu membuang kolom aslinya agar tabel lebih ringkas.

In [7]:
# 1. Isi missing value (NaN) dengan string kosong agar tidak berubah menjadi kata "nan"
df_cleaned['Subject'] = df_cleaned['Subject'].fillna('')
df_cleaned['Message'] = df_cleaned['Message'].fillna('')

# 2. Menggabungkan Subject dan Message
df_cleaned['Text'] = df_cleaned['Subject'].astype(str) + " " + df_cleaned['Message'].astype(str)

# 3. Membuang kolom Subject dan Message yang lama
df_cleaned = df_cleaned.drop(['Subject', 'Message'], axis=1)

# Melihat hasilnya
df_cleaned.head(3)

,Spam/Ham,Text
0,ham,christmas tree farm pictures
1,ham,"vastar resources , inc . gary , production fro..."
2,ham,calpine daily gas nomination - calpine daily g...


### 6. Text Preprocessing (Pembersihan Teks)
Menggunakan pustaka `re` (Regular Expression) untuk:
1. Case Folding (huruf kecil).
2. Menghapus tanda baca, angka, dan karakter khusus.
3. Menghapus spasi berlebih.

In [8]:
import re

def bersihkan_teks(teks):
    # 1. Mengubah ke huruf kecil
    teks = teks.lower()

    # 2. Hanya menyisakan huruf alfabet (a-z) dan spasi
    # (Tanda baca, angka, dan enter/newline akan diubah menjadi spasi)
    teks = re.sub(r'[^a-z\s]', ' ', teks)

    # 3. Menghapus spasi ganda yang dihasilkan dari proses di atas
    teks = re.sub(r'\s+', ' ', teks).strip()

    return teks

# Menerapkan fungsi ke seluruh baris
df_cleaned['Clean_Text'] = df_cleaned['Text'].apply(bersihkan_teks)

# Menampilkan perbandingan teks sebelum dan sesudah dibersihkan
df_cleaned[['Text', 'Clean_Text']].head()

,Text,Clean_Text
0,christmas tree farm pictures,christmas tree farm pictures
1,"vastar resources , inc . gary , production fro...",vastar resources inc gary production from the ...
2,calpine daily gas nomination - calpine daily g...,calpine daily gas nomination calpine daily gas...
3,re : issue fyi - see note below - already done...,re issue fyi see note below already done stell...
4,meter 7268 nov allocation fyi .\n- - - - - - -...,meter nov allocation fyi forwarded by lauri a ...


### 7. Stopword Removal
Menghapus kata-kata hubung atau kata umum dalam bahasa Inggris yang tidak memiliki makna signifikan untuk klasifikasi menggunakan pustaka NLTK (Natural Language Toolkit).

In [9]:
import nltk
from nltk.corpus import stopwords

# Mengunduh kamus stopword dari NLTK
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def hapus_stopword(teks):
    # Memecah kalimat menjadi daftar kata
    kata_kata = teks.split()
    # Menyaring kata yang bukan stopword
    kata_bersih = [kata for kata in kata_kata if kata not in stop_words]
    # Menggabungkannya kembali menjadi kalimat
    return ' '.join(kata_bersih)

# Menerapkan fungsi
df_cleaned['Final_Text'] = df_cleaned['Clean_Text'].apply(hapus_stopword)

# Menampilkan hasil akhir
df_cleaned[['Clean_Text', 'Final_Text']].head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,Clean_Text,Final_Text
0,christmas tree farm pictures,christmas tree farm pictures
1,vastar resources inc gary production from the ...,vastar resources inc gary production high isla...
2,calpine daily gas nomination calpine daily gas...,calpine daily gas nomination calpine daily gas...
3,re issue fyi see note below already done stell...,issue fyi see note already done stella forward...
4,meter nov allocation fyi forwarded by lauri a ...,meter nov allocation fyi forwarded lauri allen...


### 8. Menyimpan Data Bersih (Export)
Menyimpan dataset yang sudah sepenuhnya bersih ke dalam file CSV baru. Data inilah yang akan digunakan di Notebook 2 untuk melatih model.

In [10]:
# Hanya mengambil kolom Final_Text dan Spam/Ham (label)
df_final = df_cleaned[['Final_Text', 'Spam/Ham']].copy()

# Mengganti nama kolom agar lebih standar
df_final.columns = ['text', 'label']

# Menghapus baris jika ada teks yang menjadi kosong setelah stopword dihapus
df_final = df_final[df_final['text'].str.strip() != '']

# Tentukan jalur penyimpanan ke folder 'data' di Drive
save_path = '/content/drive/MyDrive/Semester 4/Machine Learning/Final Project/data/cleaned_enron_data.csv'

# Menyimpan ke CSV tanpa menyertakan index urutan
df_final.to_csv(save_path, index=False)

print(f"Data bersih berhasil disimpan di: {save_path}")
print("Dimensi data akhir yang siap dilatih:", df_final.shape)

Data bersih berhasil disimpan di: /content/drive/MyDrive/Semester 4/Machine Learning/Final Project/data/cleaned_enron_data.csv
Dimensi data akhir yang siap dilatih: (33716, 2)
